# LangGraph — آشنایی با ساختار گراف

## Outline
* LangGraph چیست و چرا به آن نیاز داریم؟
* سه مفهوم اصلی: **State** · **Node** · **Edge**
* ساخت اولین گراف ساده (بدون LLM)
* Conditional Edge — مسیریابی شرطی
* حلقه (Loop) در گراف
* Checkpointing با InMemorySaver

## ۰. نصب

```bash
pip install langgraph
```

## ۱. LangGraph چیست؟

**LangGraph** یک فریم‌ورک برای ساخت برنامه‌های AI با **جریان کنترل پیچیده** است.

```
LangChain (chain ساده):
  Input → Step1 → Step2 → Step3 → Output
  (خطی، بدون شرط، بدون حافظه)

LangGraph (گراف):
  Input → Node A
              ├─[شرط ۱]→ Node B → Node D
              └─[شرط ۲]→ Node C ↩ (حلقه)
  (شرطی، با حلقه، با حافظه پایدار)
```

### چرا LangGraph؟

| نیاز | ابزار مناسب |
|------|-------------|
| یک call ساده به LLM | `init_chat_model` |
| زنجیر خطی از مراحل | LCEL chain |
| Agent با tool-calling | `create_agent` |
| **جریان کنترل پیچیده، حلقه، تأیید انسانی** | **LangGraph** |

## ۲. سه مفهوم اصلی

```
┌─────────────────────────────────────────────┐
│                  STATE                       │
│   {"messages": [...], "counter": 3, ...}    │
│   (دیکشنری مشترک بین همه Nodeها)           │
└─────────────────────────────────────────────┘
          ↑ خواندن/نوشتن
          │
   ┌──────┴───────┐
   │    NODE A    │  ← تابع Python که State می‌گیرد
   │  (تابع)      │     و بخشی از State را آپدیت می‌کند
   └──────┬───────┘
          │
        EDGE  ← ارتباط بین Nodeها
          │    (مستقیم یا شرطی)
   ┌──────┴───────┐
   │    NODE B    │
   └──────────────┘
```

- **State**: دیکشنری مشترک — هر Node می‌خواند و می‌نویسد
- **Node**: تابع Python — `state` می‌گیرد، دیکشنری آپدیت برمی‌گرداند
- **Edge**: ارتباط — مستقیم (`add_edge`) یا شرطی (`add_conditional_edges`)

## ۳. اولین گراف — خطی (بدون LLM)

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ── ۱. تعریف State ──────────────────────────────────────
class MyState(TypedDict):
    number: int
    result: str

# ── ۲. تعریف Nodeها ─────────────────────────────────────
def double_it(state: MyState) -> dict:
    """Node اول: عدد را دو برابر می‌کند"""
    new_val = state["number"] * 2
    print(f"  [double_it]  {state['number']} × 2 = {new_val}")
    return {"number": new_val}

def describe_it(state: MyState) -> dict:
    """Node دوم: توضیح می‌نویسد"""
    text = f"عدد نهایی: {state['number']}"
    print(f"  [describe_it]  {text}")
    return {"result": text}

# ── ۳. ساخت گراف ────────────────────────────────────────
graph_builder = StateGraph(MyState)

# افزودن Nodeها
graph_builder.add_node("double", double_it)
graph_builder.add_node("describe", describe_it)

# افزودن Edgeها (جریان اجرا)
graph_builder.add_edge(START, "double")    # شروع از اینجا
graph_builder.add_edge("double", "describe")
graph_builder.add_edge("describe", END)    # پایان اینجا

# ── ۴. compile ──────────────────────────────────────────
graph = graph_builder.compile()

# ── ۵. اجرا ─────────────────────────────────────────────
print("=== اجرای گراف ===")
result = graph.invoke({"number": 5, "result": ""})
print(f"\nخروجی نهایی: {result}")

=== اجرای گراف ===
  [double_it]  5 × 2 = 10
  [describe_it]  عدد نهایی: 10

خروجی نهایی: {'number': 10, 'result': 'عدد نهایی: 10'}


### نکات مهم:

```
گراف اجرا:
  START → [double_it] → [describe_it] → END

State در هر مرحله:
  ورودی:     {number: 5,  result: ""}
  بعد double: {number: 10, result: ""}   ← فقط number آپدیت شد
  بعد describe:{number: 10, result: "عدد نهایی: 10"}  ← فقط result آپدیت شد
```

> **مهم**: Node فقط فیلدهایی که تغییر کرده را برمی‌گرداند، نه کل State را.

## ۴. Conditional Edge — مسیریابی شرطی

بعد از یک Node، **بر اساس State** تصمیم بگیریم به کجا برویم.

In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class NumberState(TypedDict):
    number: int
    label: str

# ── Nodeها ───────────────────────────────────────────────
def check_number(state: NumberState) -> dict:
    """Node اول: عدد را بررسی می‌کند"""
    print(f"  [check_number]  عدد: {state['number']}")
    return {}  # فقط routing می‌کنیم، State تغییر نمی‌کند

def handle_positive(state: NumberState) -> dict:
    return {"label": f"{state['number']} مثبت است ✓"}

def handle_negative(state: NumberState) -> dict:
    return {"label": f"{state['number']} منفی است ✗"}

def handle_zero(state: NumberState) -> dict:
    return {"label": "صفر است"}

# ── Router function ──────────────────────────────────────
# این تابع State می‌گیرد و نام Node بعدی را برمی‌گرداند
def route_number(state: NumberState) -> Literal["positive", "negative", "zero"]:
    if state["number"] > 0:
        return "positive"
    elif state["number"] < 0:
        return "negative"
    else:
        return "zero"

# ── ساخت گراف ───────────────────────────────────────────
builder = StateGraph(NumberState)

builder.add_node("check", check_number)
builder.add_node("positive", handle_positive)
builder.add_node("negative", handle_negative)
builder.add_node("zero", handle_zero)

builder.add_edge(START, "check")

# Conditional Edge: بعد از check، تابع route_number صدا زده می‌شود
builder.add_conditional_edges(
    "check",          # از کدام Node
    route_number,     # تابع router
    {
        "positive": "positive",  # اگر "positive" برگرداند → Node positive
        "negative": "negative",
        "zero":     "zero",
    }
)

builder.add_edge("positive", END)
builder.add_edge("negative", END)
builder.add_edge("zero", END)

graph = builder.compile()

# ── تست ─────────────────────────────────────────────────
for n in [7, -3, 0]:
    result = graph.invoke({"number": n, "label": ""})
    print(f"  نتیجه: {result['label']}\n")

  [check_number]  عدد: 7
  نتیجه: 7 مثبت است ✓

  [check_number]  عدد: -3
  نتیجه: -3 منفی است ✗

  [check_number]  عدد: 0
  نتیجه: صفر است



```
ساختار گراف:

  START → [check]
               ├─ number > 0 → [positive] → END
               ├─ number < 0 → [negative] → END
               └─ number = 0 → [zero]     → END
```

## ۵. Reducer — وقتی فیلد باید تجمیع شود

پیش‌فرض: آپدیت **جایگزین** می‌کند.  
با `Annotated + operator.add` می‌توانیم **اضافه** کنیم.

In [7]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class LogState(TypedDict):
    value: int
    # Annotated + operator.add → هر آپدیت اضافه می‌شود (نه جایگزین)
    log: Annotated[list[str], operator.add]

def step_a(state: LogState) -> dict:
    new_val = state["value"] + 10
    return {
        "value": new_val,
        "log": [f"step_a: {state['value']} → {new_val}"]
    }

def step_b(state: LogState) -> dict:
    new_val = state["value"] * 2
    return {
        "value": new_val,
        "log": [f"step_b: {state['value']} → {new_val}"]
    }

builder = StateGraph(LogState)
builder.add_node("a", step_a)
builder.add_node("b", step_b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

graph = builder.compile()
result = graph.invoke({"value": 5, "log": []})

print(f"مقدار نهایی: {result['value']}")
print("لاگ مراحل:")
for entry in result["log"]:
    print(f"  {entry}")

مقدار نهایی: 30
لاگ مراحل:
  step_a: 5 → 15
  step_b: 15 → 30


## ۶. Loop — حلقه در گراف

گراف می‌تواند به یک Node قبلی برگردد ← این قدرت اصلی LangGraph است.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class CounterState(TypedDict):
    count: int
    max_count: int

def increment(state: CounterState) -> dict:
    new_count = state["count"] + 1
    print(f"  [increment]  count: {state['count']} → {new_count}")
    return {"count": new_count}

def should_continue(state: CounterState) -> Literal["increment", "__end__"]:
    """Router: ادامه بده یا تمام کن؟"""
    if state["count"] < state["max_count"]:
        return "increment"  # ← برگشت به Node قبلی = حلقه
    else:
        return "__end__"    # ← END

builder = StateGraph(CounterState)
builder.add_node("increment", increment)

builder.add_edge(START, "increment")

# Conditional Edge که می‌تواند به همان Node برگردد
builder.add_conditional_edges(
    "increment",
    should_continue,
    {
        "increment": "increment",  # ← حلقه!
        "__end__": END
    }
)

graph = builder.compile()

print("=== شمارش تا ۴ ===")
result = graph.invoke({"count": 0, "max_count": 4})
print(f"\ncount نهایی: {result['count']}")

```
ساختار گراف با حلقه:

  START → [increment] ←─────────┐
               │                │
               ├─ count < max ──┘  (حلقه)
               └─ count ≥ max → END
```

> این الگو دقیقاً همان ReAct loop است که agent‌ها استفاده می‌کنند:
> فکر کن → tool بزن → نتیجه ببین → دوباره فکر کن → ...

## ۷. Checkpointing — ذخیره حالت

با `InMemorySaver` می‌توان گفتگو را از جایی که متوقف شده از سر گرفت.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class ChatState(TypedDict):
    history: Annotated[list[str], operator.add]

def add_message(state: ChatState) -> dict:
    return {}  # فقط نمایش

builder = StateGraph(ChatState)
builder.add_node("chat", add_message)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# ← checkpointer را به compile پاس می‌دهیم
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# ── thread_id = شناسه مکالمه ────────────────────────────
config = {"configurable": {"thread_id": "user_ali"}}

# پیام اول
graph.invoke({"history": ["سلام!"]}, config)

# پیام دوم — State قبلی به‌طور خودکار لود می‌شود
graph.invoke({"history": ["چطوری؟"]}, config)

# مشاهده کل State این thread
state = graph.get_state(config)
print("تاریخچه مکالمه:")
for msg in state.values["history"]:
    print(f"  {msg}")

## جمع‌بندی

```
LangGraph = StateGraph + Nodes + Edges

State      → TypedDict مشترک بین همه Nodeها
Node       → تابع Python: state می‌گیرد، dict آپدیت برمی‌گرداند
Edge       → add_edge("a", "b")  [مستقیم]
Cond. Edge → add_conditional_edges("a", router_fn, {...})  [شرطی]
Reducer    → Annotated[list, operator.add]  برای فیلدهای تجمیعی
Loop       → Conditional Edge که به Node قبلی برمی‌گردد
Checkpoint → compile(checkpointer=InMemorySaver())  برای حافظه پایدار
```

در نوت‌بوک بعدی: این ساختار را با **LLM** ترکیب می‌کنیم و یک chatbot واقعی می‌سازیم.